# Building Your First AI Agent with LangChain: From Chatbot to Agent

In this tutorial, we'll explore the evolution from simple chatbots to powerful AI agents by building a pet gift finder. You'll understand the key differences and learn when to use each approach.

## Chatbots vs AI Agents: What's the Difference?

**Chatbots** are conversational interfaces that respond to user input based on their training data. They're great for:
- Answering questions from existing knowledge
- Having conversations
- Providing explanations and advice

**AI Agents** go beyond conversation - they can take actions in the real world using tools. They can:
- Search the web for current information
- Make API calls to external services
- Perform calculations and data analysis
- Execute code and interact with databases

## What You'll Learn
- Start with a simple model (chatbot approach)
- Identify limitations of knowledge-only responses
- Build custom tools for real-world capabilities
- Create a full AI agent with LangChain
- Add multimodal capabilities (text + images)
- Deploy to LangSmith for interactive use

## Prerequisites
- Basic Python knowledge
- OpenAI API key
- Tavily API key (for web search)
- LangSmith API key (optional, for deployment)

Let's start simple and build up!

## Step 1: Environment Setup

First, we'll load our environment variables and optionally enable LangSmith tracing.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

# Optional: Enable LangSmith tracing for debugging and monitoring
# Uncomment these lines if you have LangSmith set up
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "pet-gift-finder-tutorial"

print("Environment loaded! LangSmith tracing available if configured.")

Environment loaded! LangSmith tracing available if configured.


## Step 2: Starting Simple - The Chatbot Approach

Let's begin with a basic language model to understand what chatbots can and cannot do.

In [2]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

# Initialize a basic chat model
model = init_chat_model(model="gpt-5-nano")

# Create a system message to define the chatbot's role
system_message = SystemMessage(content="""
You are a helpful pet gift advisor. Help users find Christmas gifts for their pets 
based on your knowledge of pet products and care. Provide specific product suggestions 
when possible.
""")

print("Basic Pet Gift Chatbot initialized!")

Basic Pet Gift Chatbot initialized!


### Testing the Basic Chatbot

In [3]:
# Test the basic chatbot
user_question = HumanMessage(content="I have a playful orange tabby cat. What Christmas gifts would be perfect for him?")

response = model.invoke([system_message, user_question])
print("Chatbot Response:")
print(response.content)

Chatbot Response:
Sounds like a blast to shop for! An energetic, playful cat (especially a curious orange tabby) loves gifts that spark chasing, pouncing, and mind games. Here are solid, ready-to-buy ideas with specific product suggestions:

Top picks for a playful tabby
- Da Bird Feather Teaser Wand (interactive wand toy)
  - Why: irresistible for a cat with a need to chase and pounce. Great exercise and bonding time.
  - Where: Amazon, Chewy, PetSmart
  - Price: usually around $10–$15

- PetSafe Bolt Laser Toy (automatic laser)
  - Why: gives him solo play when you’re busy and keeps him sprinting around the room.
  - Caution: supervise to avoid staring into eyes; use brief sessions.
  - Where: Amazon, Chewy, Petco
  - Price: around $20–$30

- Catit Senses 2.0 or Cat Amazing Puzzle Toy (treat puzzle)
  - Why: stimulates his brain and rewards him with treats, perfect for a smart, active cat.
  - Where: Amazon, Chewy
  - Price: roughly $20–$35 depending on model

- KONG Kickeroo (fluffy

### The Limitations Become Clear

Let's ask for something that requires current information:

In [4]:
# Ask for current pricing and availability
current_info_question = HumanMessage(content="""
What are the current prices for interactive cat toys on Amazon? 
Which ones are in stock right now and have good reviews?
""")

response = model.invoke([system_message, current_info_question])
print("Chatbot Response to Current Info Request:")
print(response.content)
print("\n" + "="*50)
print("LIMITATION: The chatbot can't access real-time information!")
print("It can only work with its training data, which has a cutoff date.")

Chatbot Response to Current Info Request:
I don’t have live access to Amazon prices or stock in this chat, so I can’t pull current numbers or confirm which items are in stock right now. I can still help you choose great interactive cat toys and show you exactly how to check prices and availability on Amazon quickly. I’ll also suggest some proven options you can search for.

How to quickly check current prices and stock on Amazon (step-by-step)
- Go to Amazon and search for “interactive cat toy” or narrow by type (laser toy, puzzle feeder, motion toy, wand toy, etc.).
- Use filters: In Stock, Prime Eligible, and sort by Avg. Customer Review or Price (Low to High or High to Low depending on your budget).
- Check the product page for: current price, “In Stock” status, Prime badge, and delivery estimates.
- Read a few recent reviews (look for 4+ stars with a few recent reviews) to gauge reliability and any safety notes.
- If you’re deciding between a few items, check “Frequently bought tog

## Step 3: Why We Need Agents

The chatbot approach has several limitations:

1. **No real-time information** - Can't check current prices, availability, or reviews
2. **No verification** - Can't confirm if products actually exist
3. **Static knowledge** - Information becomes outdated
4. **No actions** - Can't actually help you buy anything

**This is where AI agents shine!** Agents can use tools to:
- Search the web for current information
- Check real prices and availability
- Find the latest products and reviews
- Even help with purchasing (with the right integrations)

Let's build an agent that can actually help!

## Step 4: Creating Tools - The Agent's Superpowers

Now we'll give our AI the ability to take actions in the real world. Tools are Python functions that agents can call to perform specific tasks.

In [5]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

# Initialize the Tavily client for web searching
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for current information about pet gifts, products, and shopping"""
    return tavily_client.search(query)

@tool
def pet_gift_search(pet_type: str, pet_characteristics: str, budget: str = "moderate", location: str = "US") -> Dict[str, Any]:
    """Search for specific pet gifts based on type, characteristics, budget, and location"""
    search_query = f"Christmas gifts for {pet_type} {pet_characteristics} {budget} budget 2024 where to buy {location}"
    return tavily_client.search(search_query)

@tool
def local_store_search(product_name: str, location: str) -> Dict[str, Any]:
    """Find local pet stores and retailers in a specific location that might carry a product"""
    search_query = f"pet stores near {location} {product_name} in stock local retailers"
    return tavily_client.search(search_query)

print("Location-aware tools created! Our agent can now find gifts globally and locate nearby stores.")

Location-aware tools created! Our agent can now find gifts globally and locate nearby stores.


### Testing Our Tools

Let's test one of our tools directly to see how it works:

In [6]:
# Test our tool directly
test_result = web_search.invoke({"query": "best interactive cat toys 2024 Amazon"})
print("Direct tool result (first result):")
print(f"Title: {test_result['results'][0]['title']}")
print(f"URL: {test_result['results'][0]['url']}")
print(f"Content: {test_result['results'][0]['content'][:200]}...")
print("\nThis is real, current information from the web!")

Direct tool result (first result):
Title: What Are the Best Cat Toys for 2024
URL: https://cats-mode.com/blogs/cat-thing-blog/what-are-the-best-cat-toys-for-2024?srsltid=AfmBOooDn6zqLOHt9gTcGqxhag3JtaKH3hELKie7vwg-6SNrJqLwWNDg
Content: What Are the Best Cat Toys for 2024 · YVE LIFE Automatic Cat Laser Toy - $25 at Amazon · Potaroma Electric Flopping Fish – $12.98-$13.99 at Amazon....

This is real, current information from the web!


## Step 5: Designing the Agent's System Prompt

The system prompt is crucial for agents - it needs to explain not just the role, but also how to use tools effectively.

In [7]:
system_prompt = """
You are a helpful pet gift advisor with access to real-time web search capabilities.

Your role:
- Help pet owners find appropriate Christmas gifts based on their pet's characteristics
- Use your web search tools to find current products, prices, and availability
- Consider pet safety, size, age, and personality when making recommendations
- Provide specific product suggestions with purchasing information
- Offer options across different budget ranges
- Help users find local stores and retailers in their area

When to use tools:
- Use web_search for general queries about pet products, reviews, or shopping
- Use pet_gift_search when you have specific pet characteristics, budget, and location info
- Use local_store_search to find nearby pet stores that might carry specific products
- Always ask for the user's location if they want local shopping options
- Always search for current information rather than relying on outdated knowledge

Location examples:
- For US users: Search Amazon, Petco, PetSmart, Chewy
- For Philippines users: Search Shopee, Lazada, local pet stores in Manila/Cebu/Davao
- For other countries: Adapt to local e-commerce and pet store chains

When analyzing pets from photos:
- Identify breed characteristics that might influence gift choices
- Estimate size and age if possible
- Note any visible personality traits or energy levels

Always prioritize pet safety and provide real, purchasable products with current pricing and local availability.
"""

print("System prompt created with location-aware instructions!")

System prompt created with location-aware instructions!


## Step 6: Creating the AI Agent

Now we'll combine everything into a powerful AI agent:

In [8]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Create our AI agent with tools
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, pet_gift_search, local_store_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()  # Enables conversation memory
)

print("AI Agent created! It now has location-aware search capabilities.")

AI Agent created! It now has location-aware search capabilities.


## Step 7: Adding Streaming Responses

Before we test our agent, let's add streaming capabilities for a better user experience. Streaming shows responses as they're generated, making the interaction feel more natural and responsive.

In [ ]:
def stream_agent_response(message, config, show_thinking=True):
    """Stream the agent's response for better user experience"""
    if show_thinking:
        print("🤖 Agent is thinking and searching...\n")
    
    response_content = ""
    
    # Stream the response
    for chunk in agent.stream({"messages": [HumanMessage(content=message)]}, config):
        # Handle different types of chunks
        if "messages" in chunk and chunk["messages"]:
            last_message = chunk["messages"][-1]
            if hasattr(last_message, 'content') and last_message.content:
                # Only print new content (avoid duplicates)
                new_content = last_message.content
                if new_content != response_content:
                    # Print only the new part
                    print(new_content[len(response_content):], end="", flush=True)
                    response_content = new_content
        
        # Show tool usage for transparency
        elif "tools" in chunk:
            for tool_call in chunk["tools"]:
                if hasattr(tool_call, 'name'):
                    print(f"\n🔍 Using tool: {tool_call.name}\n", flush=True)
    
    print("\n\n✅ Response complete!")
    return response_content

print("Streaming function created! Now responses will appear in real-time.")

## Step 8: Testing the Agent with Streaming

Let's test our agent with streaming responses for a better user experience:

In [ ]:
from langchain.messages import HumanMessage

# Configuration for conversation memory
config = {"configurable": {"thread_id": "pet_gift_session_1"}}

# Test with streaming - same question we asked the basic chatbot
print("🐱 Question: I have a playful orange tabby cat who loves to chase things and knock items off tables. What Christmas gifts would be perfect for him?\n")

stream_agent_response(
    "I have a playful orange tabby cat who loves to chase things and knock items off tables. What Christmas gifts would be perfect for him?",
    config
)

### Comparing: Non-Streaming vs Streaming

Let's also show the traditional non-streaming approach for comparison:

In [ ]:
# Traditional non-streaming approach (for comparison)
print("📝 Traditional Non-Streaming Response:\n")

response = agent.invoke(
    {"messages": [HumanMessage(content="What are some budget-friendly cat toys under $20?")]},
    config
)

print("AI Agent Response (traditional):")
print(response['messages'][-1].content)
print("\n" + "="*50)
print("Notice: Traditional responses appear all at once after processing is complete.")
print("Streaming responses appear progressively, providing better user experience!")

### Testing Current Information with Streaming - The Agent's Superpower

In [9]:
from langchain.messages import HumanMessage

# Configuration for conversation memory
config = {"configurable": {"thread_id": "pet_gift_session_1"}}

# Ask the same question we asked the basic chatbot
response = agent.invoke(
    {"messages": [HumanMessage(content="I have a playful orange tabby cat who loves to chase things and knock items off tables. What Christmas gifts would be perfect for him?")]},
    config
)

print("AI Agent Response (with real-time web search):")
print(response['messages'][-1].content)

AI Agent Response (with real-time web search):
That’s a great profile for a Christmas gift plan! An energetic, playful orange tabby who loves to chase and knock things off tables will do best with a mix of chase-focused toys, brain-teasers, and sturdier perches or posts. Here are real, purchasable ideas across budgets, with quick notes on why they fit and where to buy.

Top picks for your chase-happy cat

1) Interactive treat puzzle toy (brain teaser + reward)
- Cat Amazing Interactive Treat Maze & Puzzle Cat Toy
- Why it fits: challenges him to “hunt” for treats, satisfying his chasing instincts without turning your whole living room into a battlefield. Great for mental stimulation and slow feeding/snacks.
- Where to buy and price: Chewy listing shows around $13-$18 depending on version.
- Quick tip: start with easy levels and rotate once he’s good at it to keep him engaged.

2) Wand teaser toy (classic for chase, great for redirection)
- MeoHui Retractable Cat Wand Toy (often recomme

### Testing Current Information - The Agent's Superpower

In [10]:
# Test current information with streaming - the question that stumped the chatbot
print("💰 Question: What are the current prices for interactive cat toys on Amazon? Which ones are in stock right now and have good reviews?\n")

stream_agent_response(
    "What are the current prices for interactive cat toys on Amazon? Which ones are in stock right now and have good reviews?",
    config
)

print("\n" + "="*50)
print("🎉 SUCCESS: The agent can access real-time information with streaming!")
print("You can see the agent thinking, using tools, and building the response in real-time.")

AI Agent Response to Current Info Request:
I can pull live prices, stock status, and ratings from Amazon for your region, but I’ll need your location to do it accurately. Do you want US Amazon data (or another country)? If you share your country/ZIP, I’ll fetch current, in-stock options and rank them by reviews and price.

Here’s a quick snapshot of current price cues and typical stock/status you’ll often see for popular options (based on the latest data I pulled):

- Cat Amazing HEX (Interactive Treat Puzzle Box)
  - Typical price: about $29.95 on the CatAmazing site.
  - Amazon price range: usually around $24–40 depending on seller and promos.
  - Stock/reviews: Generally well-reviewed on Amazon; usually in stock, but can vary by seller.

- Cat Amazing MEGA
  - Typical price: about $34.95 on the CatAmazing site.
  - Amazon price range: roughly in the mid-$20s to mid-$40s depending on promotions.
  - Stock/reviews: Strong reviews on Amazon; commonly available but can fluctuate.

- ROJ

## Step 9: Location-Aware Shopping with Streaming

Let's test the location-aware capabilities with streaming responses for different regions:

In [11]:
# Example for Philippines users with streaming
print("🇵🇭 Question: I'm in Manila, Philippines and have a small Shih Tzu. What Christmas gifts can I find locally or on Filipino e-commerce sites like Shopee or Lazada? Budget is around 1000-2000 PHP.\n")

stream_agent_response(
    "I'm in Manila, Philippines and have a small Shih Tzu. What Christmas gifts can I find locally or on Filipino e-commerce sites like Shopee or Lazada? Budget is around 1000-2000 PHP.",
    config
)

Philippines Shopping Response:
Fantastic—Manila has plenty of budget-friendly options on Shopee, Lazada, and at local pet stores. With 1000–2000 PHP, you can put together a cute, useful Christmas gift bundle for your small Shih Tzu or grab a nice single item.

Here are practical picks you can actually buy locally right now, with price hints and buying links:

1) Nina Ottosson Snack Palz Interactive Plush Dog Puzzle Toy
- Why it’s good: mentally engaging for a small dog; rewards him with treats as he “solves” the puzzle.
- Price (approx): around PHP 495 (often on sale at local shops).
- Where to buy: Pet Warehouse Philippines
- Link: https://www.petwarehouse.ph/dog/nina-ottosson-snack-palz-interactive-plush-dog-puzzle-toy-with-treat-ball.html
- Budget fit: great starter puzzle toy that leaves room for another small gift.

2) Tough Squeaky Plush Toys for Small to Large Breeds (plush chew/toy)
- Why it’s good: soft, satisfying for a small dog, good for fetch/throw-and-tounce play; easy on

In [12]:
# Example for finding local stores with streaming
print("🏪 Question: I found a great interactive puzzle feeder online, but I'd prefer to buy it locally. I'm in Austin, Texas. Can you help me find pet stores nearby that might carry puzzle feeders?\n")

stream_agent_response(
    "I found a great interactive puzzle feeder online, but I'd prefer to buy it locally. I'm in Austin, Texas. Can you help me find pet stores nearby that might carry puzzle feeders?",
    config
)

Local Store Finder Response:
Absolutely. Here are Austin-area stores that typically carry interactive puzzle feeders or dog-treat puzzle toys, with notes on what they stock and how to check current availability.

Stores you can try
- Hollywood Feed (Burnet Rd)
  - What they stock: a range of dog toys, including interactive/puzzle toys and feeders. Staff are usually helpful with choosing a puzzle feeder.
  - Location: 4604 Burnet Rd, Austin, TX
  - Why check them: dedicated pet store with good selection and frequent stock refreshes.
  - Quick tip: call ahead to confirm they have a specific puzzle feeder model in stock.

- Tomlinson’s Feed (Austin area)
  - What they stock: curated “interactive/dog smart toys” collection; often includes puzzle feeders or feeders that promote mental stimulation.
  - How to shop: in-store or curbside pickup; they list Austin-area availability on their site.
  - Quick tip: ask staff to point you to puzzle feeders or treat-dispensing toys.

- PetSmart (multi

## Step 10: Advanced Streaming Example

Let's test streaming with a more complex query that will require multiple tool calls:

In [13]:
# Test streaming with a complex query that requires multiple searches
print("🐕 Complex Question: I have a senior dog (12 years old) with arthritis. What Christmas gifts would help with his comfort and mobility? Please include current prices and where to buy them.\n")

stream_agent_response(
    "I have a senior dog (12 years old) with arthritis. What Christmas gifts would help with his comfort and mobility? Please include current prices and where to buy them.",
    config
)

Agent is thinking and searching...



Response complete!


## Step 11: Adding Multimodal Capabilities

Let's add image analysis so users can upload photos of their pets:

In [14]:
from ipywidgets import FileUpload
from IPython.display import display
import base64

uploader = FileUpload(
    accept='.png,.jpg,.jpeg', 
    multiple=False,
    description='Upload Pet Photo'
)
display(uploader)

FileUpload(value=(), accept='.png,.jpg,.jpeg', description='Upload Pet Photo')

In [15]:
def process_uploaded_image():
    """Convert uploaded image to base64 format for the AI model"""
    if not uploader.value:
        return None, None
    
    # Get the uploaded file
    uploaded_file = uploader.value[0]
    
    # Convert memoryview to bytes
    content_mv = uploaded_file["content"]
    img_bytes = bytes(content_mv)
    
    # Base64 encode for the model
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    
    return img_b64, uploaded_file["type"]

# Process the image
img_b64, file_type = process_uploaded_image()

if img_b64:
    print("Image processed successfully and ready for analysis!")
else:
    print("Please upload an image first.")

Please upload an image first.


In [16]:
if img_b64:
    # Create a multimodal message with both text and image
    multimodal_message = HumanMessage(content=[
        {
            "type": "text", 
            "text": "Here's a photo of my pet! Please analyze their appearance, size, breed characteristics, and any personality traits you can observe, then recommend Christmas gifts that would be perfect for them. Include specific products and where to buy them."
        },
        {
            "type": "image", 
            "base64": img_b64, 
            "mime_type": file_type
        }
    ])
    
    # Send to our agent
    response = agent.invoke(
        {"messages": [multimodal_message]},
        config
    )
    
    print(response['messages'][-1].content)
else:
    print("Please upload a pet photo first!")

Please upload a pet photo first!


## Step 12: Monitoring with LangSmith (Optional)

LangSmith provides powerful tracing and debugging capabilities for your agents. If you have LangSmith set up, you can monitor:

- **Tool usage**: See which tools your agent calls and why
- **Performance metrics**: Track response times and token usage
- **Debugging**: Inspect the full reasoning chain when things go wrong
- **User feedback**: Collect ratings and improve your agent over time

To enable LangSmith tracing, uncomment the lines in Step 1 and set your LangSmith API key in your `.env` file:

```
LANGCHAIN_API_KEY=your_langsmith_api_key_here
```

Once enabled, you can view traces at [smith.langchain.com](https://smith.langchain.com)

## Key Learnings: Chatbots vs Agents + Streaming Benefits

Through this tutorial, we've seen the evolution from simple chatbots to powerful AI agents with streaming capabilities:

### Chatbots (Step 2)
- ✅ Good for: Conversations, explanations, advice based on training data
- ❌ Limited by: Static knowledge, no real-time information, can't take actions

### AI Agents (Steps 4-13)
- ✅ Can: Search the web, find current prices, locate nearby stores, process images
- ✅ Provide: Real-time information, location-aware recommendations, actionable results
- ✅ Scale: Add new tools easily, monitor performance, stream responses

### Streaming Benefits (Steps 7-10)
- ✅ **Better UX**: Users see responses as they're generated, not all at once
- ✅ **Transparency**: Shows when tools are being used and what the agent is thinking
- ✅ **Engagement**: Keeps users engaged during longer processing times
- ✅ **Real-time feedback**: Users can see the agent working through complex queries

### When to Use Each
- **Use chatbots** for: FAQ systems, educational content, creative writing
- **Use agents** for: Shopping assistance, research tasks, data analysis, real-world actions
- **Use streaming** for: Any agent interaction where response time > 2-3 seconds

The key insight: **Streaming Agents = Chatbots + Tools + Real-world capabilities + Better UX!**

## Step 13: Deploying to LangSmith

To deploy this agent to LangSmith for interactive use:

1. Make sure your `.env` file has all required API keys
2. Use the provided `langgraph_pet_gift.json` configuration file
3. Deploy with: `langgraph deploy --config langgraph_pet_gift.json`

Your agent will then be available through LangSmith's web interface for interactive chat!